In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
import json

# Helper function to classify departure time
def classify_time(dep_time):
    time_obj = datetime.strptime(dep_time, '%H:%M')
    if time_obj.hour < 11:
        return "Morning"
    elif 11 <= time_obj.hour < 17:
        return "Afternoon"
    else:
        return "Evening"

# Part 1: Fetch flight prices for one day (Mumbai to Delhi, one week from today)
def fetch_one_day_flight_data():
    url = "https://api.example.com/flights"  # Replace with a real flight API endpoint
    departure_date = (datetime.today() + timedelta(days=7)).strftime('%Y-%m-%d')
    params = {
        "from": "BOM",  # Mumbai
        "to": "DEL",    # Delhi
        "date": departure_date,
        "class": "economy",
        "nonstop": "true"
    }
    headers = {
        "Authorization": "Bearer your_api_key",  # Replace with your API key
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0.4472.124"
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=5)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = response.json()

        # Assuming API returns a list of flights
        flights = []
        for flight in data.get("flights", []):
            flights.append({
                "Airline": flight.get("airline", ""),
                "Departure Time": flight.get("departure_time", ""),
                "Price": float(flight.get("price", 0))
            })

        # Save raw data to Excel
        df = pd.DataFrame(flights)
        df["Time Category"] = df["Departure Time"].apply(classify_time)
        df.to_excel("raw_flight_data.xlsx", index=False)

        # Aggregate by airline and time category
        summary = df.groupby(["Airline", "Time Category"])["Price"].mean().reset_index()
        summary.to_excel("summary_by_airline.xlsx", index=False)

        return flights

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
    except requests.exceptions.ConnectionError:
        print("Connection Error: Check your network")
    except requests.exceptions.Timeout:
        print("Request timed out")
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
    return []

# Part 2: Fetch one-month daily flight data and remove noise
def fetch_monthly_flight_data():
    url = "https://api.example.com/flights"
    headers = {
        "Authorization": "Bearer your_api_key",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0.4472.124"
    }
    all_flights = []

    # Loop over 30 days
    for i in range(30):
        current_date = datetime.today() + timedelta(days=i)
        departure_date = (current_date + timedelta(days=7)).strftime('%Y-%m-%d')
        params = {
            "from": "BOM",
            "to": "DEL",
            "date": departure_date,
            "class": "economy",
            "nonstop": "true"
        }

        try:
            response = requests.get(url, params=params, headers=headers, timeout=5)
            response.raise_for_status()
            data = response.json()

            for flight in data.get("flights", []):
                all_flights.append({
                    "Date": departure_date,
                    "Airline": flight.get("airline", ""),
                    "Departure Time": flight.get("departure_time", ""),
                    "Price": float(flight.get("price", 0))
                })

        except requests.exceptions.RequestException as e:
            print(f"Error on {departure_date}: {e}")
        time.sleep(1)  # Avoid overwhelming the API

    # Save raw data
    df = pd.DataFrame(all_flights)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Time Category"] = df["Departure Time"].apply(classify_time)
    df.to_excel("raw_flight_data_month.xlsx", index=False)

    # Remove noise (exclude weekends and holidays)
    df["Day"] = df["Date"].dt.day_name()
    holidays = ["2025-08-15"]  # Example: Independence Day
    df_cleaned = df[~df["Day"].isin(["Saturday", "Sunday"]) & ~df["Date"].isin(holidays)]

    # Aggregate monthly data
    summary = df_cleaned.groupby(["Airline", "Time Category"])["Price"].median().reset_index()
    summary.to_excel("summary_by_airline_month.xlsx", index=False)

# Example: Downloading a file (e.g., flight schedule PDF)
def download_flight_schedule():
    url = "https://example.com/flight_schedule.pdf"
    try:
        response = requests.get(url, stream=True, timeout=5)
        response.raise_for_status()
        with open("flight_schedule.pdf", "wb") as f:
            f.write(response.content)
        print("File downloaded successfully")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")

# Example: Using sessions for authentication
def login_and_fetch_data():
    session = requests.Session()
    login_url = "https://api.example.com/login"
    data = {
        "username": "user",
        "password": "pass"
    }
    try:
        response = session.post(login_url, json=data, timeout=5)
        response.raise_for_status()
        print("Login successful")

        # Fetch protected data
        protected_url = "https://api.example.com/protected_data"
        response = session.get(protected_url, timeout=5)
        print(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")

# Run the examples
if __name__ == "__main__":
    print("Fetching one-day flight data...")
    fetch_one_day_flight_data()
    print("Fetching monthly flight data...")
    fetch_monthly_flight_data()
    print("Downloading flight schedule...")
    download_flight_schedule()
    print("Testing session-based authentication...")
    login_and_fetch_data()